## Pyspark used .groupBy method to group the data (specific columns).Followed by Agregate Function like sum, avg, count etc.

In [141]:
from pyspark.sql import SparkSession
from  pyspark.sql import functions as F


In [142]:
spark=SparkSession.builder.appName("GroupByDemo").getOrCreate()

create a dummy data

In [143]:
data=[
    ("IT","Gourav",100000),
    ("Sales","Amit",50000),
    ("IT","Divya",200000),
    ("Marketing","Dev",45000),
    ("Sales","Aman",56000)
]
schema=['department','name','salary']

df=spark.createDataFrame(data,schema)

In [144]:
df.show()

+----------+------+------+
|department|  name|salary|
+----------+------+------+
|        IT|Gourav|100000|
|     Sales|  Amit| 50000|
|        IT| Divya|200000|
| Marketing|   Dev| 45000|
|     Sales|  Aman| 56000|
+----------+------+------+



##Simple Agreegate and Group by

Budget for Each Department

In [145]:
df.groupBy("department").sum("salary").show()

+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|     Sales|     106000|
|        IT|     300000|
| Marketing|      45000|
+----------+-----------+



###Average salary of Employee in Each Department

In [146]:
df.groupBy("department").avg("salary").show()

+----------+-----------+
|department|avg(salary)|
+----------+-----------+
|     Sales|    53000.0|
|        IT|   150000.0|
| Marketing|    45000.0|
+----------+-----------+



###For Advance Creation we can use .agg() function . it allows to run multiple calculation as once and rename the output for cleaner look

In [147]:
df_summary=df.groupBy('department').agg(
    F.sum('salary').alias('TotalSalary'),
    F.avg('salary').alias('AvgSalary'),
    F.count('name').alias('EmployeeCount'),
    F.max('salary').alias('MaximumSalary'),

)

In [148]:
df_summary.show()

+----------+-----------+---------+-------------+-------------+
|department|TotalSalary|AvgSalary|EmployeeCount|MaximumSalary|
+----------+-----------+---------+-------------+-------------+
|     Sales|     106000|  53000.0|            2|        56000|
|        IT|     300000| 150000.0|            2|       200000|
| Marketing|      45000|  45000.0|            1|        45000|
+----------+-----------+---------+-------------+-------------+



In [149]:
df_summary.filter(F.col("Totalsalary")>50000).show()

+----------+-----------+---------+-------------+-------------+
|department|TotalSalary|AvgSalary|EmployeeCount|MaximumSalary|
+----------+-----------+---------+-------------+-------------+
|     Sales|     106000|  53000.0|            2|        56000|
|        IT|     300000| 150000.0|            2|       200000|
+----------+-----------+---------+-------------+-------------+



In [150]:
df.groupBy('department','salary').count().show()

+----------+------+-----+
|department|salary|count|
+----------+------+-----+
|     Sales| 50000|    1|
|        IT|100000|    1|
|        IT|200000|    1|
|     Sales| 56000|    1|
| Marketing| 45000|    1|
+----------+------+-----+



# Conditional Aggregation with PySpark

Here we will use GroupBy method along with Aggregate functions like sum,count,etc. with F.when() methods.

In [151]:
from pyspark.sql import SparkSession
from  pyspark.sql import functions as F


In [152]:
spark=SparkSession.builder.appName("AggregateDemo").getOrCreate()

This data stores about Online or In-store Sales.Amount could be positive(buy) or negative (return)

In [153]:
data=[
    ("Store_A","Online",150),
    ("Store_A","In-store",50),
    ("Store_A","Online",300),
    ("Store_B","In-store",400),
    ("Store_B","Online",-100),
    ("Store_B","In-store",200),
]
columns=["Store", "Type","Amount"]

In [154]:
df=spark.createDataFrame(data,columns)
df.show()

+-------+--------+------+
|  Store|    Type|Amount|
+-------+--------+------+
|Store_A|  Online|   150|
|Store_A|In-store|    50|
|Store_A|  Online|   300|
|Store_B|In-store|   400|
|Store_B|  Online|  -100|
|Store_B|In-store|   200|
+-------+--------+------+



To use agrregate Function only on specific rows, we use F.when(condition,values)

In [155]:
df_conditional=df.groupBy("Store").agg(
    F.sum(F.when(F.col("Amount")>0, F.col("Amount"))).alias("TotalSales"),
    F.count(F.when(F.col("Type")=="Online",1)).alias("OnlineTransactionCount"),
    F.sum(F.when(F.col("Amount")<0, F.col("Amount"))).alias("Total_Return_Sales")
  )

In [156]:
df_conditional.show()

+-------+----------+----------------------+------------------+
|  Store|TotalSales|OnlineTransactionCount|Total_Return_Sales|
+-------+----------+----------------------+------------------+
|Store_A|       500|                     2|              NULL|
|Store_B|       600|                     1|              -100|
+-------+----------+----------------------+------------------+



###We Do Statical Analysis

In [157]:
df_analysis=df.groupBy("Store").agg(
    F.avg(F.when(F.col("Type")=="Online",F.col("Amount"))).alias("AvgOnlineAmount"),##MEAN Amount for Online Orders Only
    (F.count(F.when(F.col("Type")=="In-store",1))/F.count("*") * 100).alias("In_StorePercentage")
    ## Conditional Percentage: (In-Store Transaction / Total Transaction  ) * 100
    )


In [158]:
df_analysis.show()

+-------+---------------+------------------+
|  Store|AvgOnlineAmount|In_StorePercentage|
+-------+---------------+------------------+
|Store_A|          225.0| 33.33333333333333|
|Store_B|         -100.0| 66.66666666666666|
+-------+---------------+------------------+



Use .na.fill method (data imputation) for null values

In [159]:
df_cleaned=df_conditional.na.fill(0,["Total_Return_Sales"])
df_cleaned.show()

+-------+----------+----------------------+------------------+
|  Store|TotalSales|OnlineTransactionCount|Total_Return_Sales|
+-------+----------+----------------------+------------------+
|Store_A|       500|                     2|                 0|
|Store_B|       600|                     1|              -100|
+-------+----------+----------------------+------------------+

